In [16]:
#auto reload
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gym
import d4rl
from d4rl_world_model import D4RLWorldModel, MLPNetwork

import sys
import os
sys.path.append("..")
from common.normalizer import StandardNormalizer
from common import util



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
env_name = "halfcheetah-random-v0"

cheetah_model = D4RLWorldModel(env_name)


/home/ubuntu/miniconda3/envs/py3/lib/python3.9/site-packages/gym/envs/registration.py:564: UserWarning: WARN: The environment halfcheetah-random-v0 is out of date. You should consider upgrading to version `v2`.
  logger.warn(
/home/ubuntu/miniconda3/envs/py3/lib/python3.9/site-packages/gym/envs/mujoco/mujoco_env.py:46: UserWarning: WARN: This version of the mujoco environments depends on the mujoco-py bindings, which are no longer maintained and may stop working. Please upgrade to the v4 versions of the environments (which depend on the mujoco python bindings instead), unless you are trying to precisely replicate previous works).
  logger.warn(
/home/ubuntu/miniconda3/envs/py3/lib/python3.9/site-packages/d4rl/gym_mujoco/gym_envs.py:18: UserWarning: This environment is deprecated. Please use the most recent version of this environment.
  offline_env.OfflineEnv.__init__(self, **kwargs)
/home/ubuntu/miniconda3/envs/py3/lib/python3.9/site-packages/gym/spaces/box.py:112: UserWarning: WARN: 

loaded dataset


In [19]:
ls ../saved_models/halfcheetah-random-v0/

world_model_0.00.pth


In [23]:
model_path = "../saved_models/halfcheetah-random-v0/world_model_0.00.pth"
cheetah_model.load_model(model_path)

/home/ubuntu/abiomed/mbpo_uq/models/d4rl_world_model.py:146: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.input_dim = obs_dim + action_dim


In [ ]:

crps = scoring.crps_evaluation(np.array(mult_pred[:,i,:,0]), np.array(target[i,:,0]))

In [ ]:
import scoring
import tqdm

dataset = cheetah_model.dataset
obs_ = []
next_obs_ = []
action_ = []
full_action_ = []
reward_ = []
terminal_ = []

crps_list = []
for i in range(len(dataset['observations'])):
    state = dataset['observations'][i]
    action = dataset['actions'][i]
    next_state = dataset['next_observations'][i]
    reward = dataset['rewards'][i]
    terminal = dataset['terminals'][i]

    pred_input = torch.FloatTensor(np.concatenate([state, action])).to(cheetah_model.device)
    model_pred = cheetah_model.model.predict_multiple(pred_input, num_samples=50).cpu().numpy()
    print(model_pred.shape, next_state.shape)

    # get crps from cheetah_model
    crps = scoring.crps_evaluation(model_pred, next_state)
    crps_list.append(crps.mean())




(50, 17) (17,)


In [34]:
crps_list

[array([0.07506411, 0.01085244, 0.20736611, 0.33636004, 0.11470053,
        0.24849056, 0.15429197, 0.02797368, 0.20156898, 1.0864525 ,
        1.3254545 , 4.115062  , 9.073241  , 0.9212824 , 6.9257627 ,
        0.303177  , 0.8619059 ], dtype=float32)]